# Build and test the crop container

This notebook is linked to https://eoap.github.io/mastering-app-package/containers/crop/

## Goal

Create a container and run the crop step in the container image.


## Setup the environment

In [5]:
export WORKSPACE=/workspace/mastering-app-package
export RUNTIME=${WORKSPACE}/runs
rm -fr ${WORKSPACE}/runs
mkdir -p ${RUNTIME}
cd ${RUNTIME}

## Build the container

Inspect the container file:

In [6]:
cat ${WORKSPACE}/water-bodies/command-line-tools/crop/Dockerfile

FROM rockylinux/rockylinux:10.2-minimal AS builder

RUN microdnf -y update && \
    microdnf -y install curl tar python3 python3-pip python3-setuptools gcc && \
    microdnf clean all

RUN curl -L https://github.com/pypa/hatch/releases/download/hatch-v1.14.0/hatch-x86_64-unknown-linux-gnu.tar.gz \
      -o /tmp/hatch.tar.gz && \
    tar -xzf /tmp/hatch.tar.gz -C /tmp && \
    install -m 0755 /tmp/hatch /usr/local/bin/hatch && \
    rm -rf /tmp/hatch* /tmp/hatch.tar.gz

WORKDIR /src
COPY . /src

# Build a wheel for your project (outputs into /src/dist)
RUN hatch build -t wheel

FROM rockylinux/rockylinux:10.2-minimal AS runtime


RUN microdnf -y update && \
microdnf -y install python3 python3-pip git && \
microdnf clean all

COPY --from=builder /usr/local/bin/hatch /usr/bin/hatch

# Non-root user
ENV HOME=/home/neo
RUN /usr/sbin/groupadd -g 2000 neo && \
    /usr/sbin/useradd -u 2000 -g 2000 -m -d ${HOME} -s /sbin/nologin neo && \
    mkdir -p /app && \
    chown -R 2000:2000 /app ${HOM

Build the container using `podman`:

In [7]:
podman build --format docker -t localhost/crop:latest ${WORKSPACE}/water-bodies/command-line-tools/crop


[1/2] STEP 1/6: FROM rockylinux/rockylinux:10.2-minimal AS builder
Resolved "rockylinux/rockylinux" as an alias (/home/fbrito/.cache/containers/short-name-aliases.conf)
Trying to pull docker.io/rockylinux/rockylinux:10.2-minimal...
Getting image source signatures
Copying blob 639b8cac9893 [-------------------------------------] 0.0b / 51.1MiB
Copying blob 639b8cac9893 [-------------------------------------] 0.0b / 51.1MiB
Copying blob 639b8cac9893 [---------------------------] 0.0b / 51.1MiB | 0.0 b/s
Copying blob 639b8cac9893 [>---------------------] 2.6MiB / 51.1MiB | 29.9 MiB/s
Copying blob 639b8cac9893 [==>-------------------] 6.0MiB / 51.1MiB | 39.3 MiB/s
Copying blob 639b8cac9893 [===>------------------] 9.8MiB / 51.1MiB | 51.0 MiB/s
Copying blob 639b8cac9893 [====>----------------] 13.4MiB / 51.1MiB | 44.4 MiB/s
Copying blob 639b8cac9893 [======>--------------] 16.8MiB / 51.1MiB | 36.4 MiB/s
Copying blob 639b8cac9893 [=======>-------------] 19.9MiB / 51.1MiB | 49.7 MiB/s
Copying

Show the `crop` help:

In [8]:
podman run --rm -it --env=PYTHONPATH=/app localhost/crop:latest crop --help

Usage: crop [OPTIONS]

  No info provided

Options:
  --input-item TEXT  [default: (item); required]
  --aoi TEXT         [default: (aoi); required]
  --epsg TEXT        [default: (epsg); required]
  --band TEXT        [default: (band); required]
  --help             Show this message and exit.


## Test the crop step in the container

Crop the green band asset:

In [10]:
podman run \
    -i \
    --userns=keep-id \
    --mount=type=bind,source=/workspace/mastering-app-package/runs,target=/runs \
    --workdir=/runs \
    --read-only=true \
    --user=1001:100 \
    --rm \
    --env=HOME=/runs \
    localhost/crop:latest \
    crop \
    --aoi \
    '{"type":"Polygon","coordinates":[[[-121.399,39.834],[-120.74,39.834],[-120.74,40.472],[-121.399,40.472],[-121.399,39.834]]]}' \
    --band \
    green \
    --epsg \
    "EPSG:4326" \
    --input-item \
    https://earth-search.aws.element84.com/v0/collections/sentinel-s2-l2a-cogs/items/S2B_10TFK_20210713_0_L2A

2026-09-23 10:10:47.854 | INFO     | crop.crop_impl:execute:77 - Starting crop for band green
2026-09-23 10:10:47.855 | INFO     | crop.crop_impl:execute:81 - Parsed Polygon AOI in EPSG:4326
2026-09-23 10:10:47.855 | INFO     | crop.crop_impl:_read_item:22 - Reading input STAC document
2026-09-23 10:10:48.638 | INFO     | crop.crop_impl:_read_item:25 - Loaded STAC item S2B_10TFK_20210713_0_L2A
2026-09-23 10:10:48.639 | INFO     | crop.crop_impl:_asset:69 - Selected asset B03 for band green
2026-09-23 10:10:48.639 | INFO     | crop.crop_impl:execute:90 - Opening raster for item S2B_10TFK_20210713_0_L2A, band green
2026-09-23 10:10:51.377 | INFO     | crop.crop_impl:execute:94 - Source raster: 10980 x 10980 pixels, 1 band(s), CRS EPSG:32610
2026-09-23 10:10:51.377 | INFO     | crop.crop_impl:execute:101 - Transforming AOI from EPSG:4326 to EPSG:32610
2026-09-23 10:10:51.380 | INFO     | crop.crop_impl:execute:103 - Cropping raster to the AOI
2026-09-23 10:11:17.720 | INFO     | crop.crop

: 1

Crop the nir band asset:

In [ ]:
podman run \
    -i \
    --userns=keep-id \
    --mount=type=bind,source=/workspace/mastering-app-package/runs,target=/runs \
    --workdir=/runs \
    --read-only=true \
    --user=1001:100 \
    --rm \
    --env=HOME=/runs \
    localhost/crop:latest \
    crop \
    --aoi \
    '{"type":"Polygon","coordinates":[[[-121.399,39.834],[-120.74,39.834],[-120.74,40.472],[-121.399,40.472],[-121.399,39.834]]]}' \
    --band \
    nir \
    --epsg \
    "EPSG:4326" \
    --input-item \
    https://earth-search.aws.element84.com/v0/collections/sentinel-s2-l2a-cogs/items/S2B_10TFK_20210713_0_L2A

List the outputs:

In [ ]:
tree ${RUNTIME}